# Recommandation et GCN

Deux jouets contrôlés : interactions implicites puis graphe assortatif. Les évaluations sont respectivement un masquage aléatoire et un protocole transductif. Elles ne remplacent pas une évaluation temporelle de production.

## 1. Interactions et item masqué

Chaque utilisateur a un groupe de préférences. On masque un item pertinent avant de calculer les similarités, puis on exclut les items déjà vus des candidats.

In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
torch.set_num_threads(1)
rng = np.random.default_rng(42)
users, items = 120, 30
interactions = np.zeros((users,items))
held = []
for u in range(users):
    liked = rng.choice(np.arange((u%3)*10,(u%3+1)*10),6,replace=False)
    held.append(liked[0]); interactions[u,liked[1:]] = 1
held = np.array(held)
assert np.all(interactions[np.arange(users),held] == 0)
norm = np.linalg.norm(interactions,axis=0)
similarity = interactions.T@interactions/np.maximum(np.outer(norm,norm),1e-12)
np.fill_diagonal(similarity,0)
collaborative = interactions@similarity
popular = np.broadcast_to(interactions.sum(0),interactions.shape).copy()
def ranking_metrics(scores,k=5):
    scores = scores.copy(); scores[interactions > 0] = -np.inf
    ranked = np.argsort(-scores,axis=1,kind='stable')[:,:k]
    rel = ranked == held[:,None]
    assert not np.any(interactions[np.arange(users)[:,None],ranked])
    return {'Recall@5':rel.any(1).mean(), 'NDCG@5':(rel/np.log2(np.arange(k)+2)).sum(1).mean()}
print(pd.DataFrame({'popularité':ranking_metrics(popular), 'similarité items':ranking_metrics(collaborative)}).T)

                  Recall@5    NDCG@5
popularité        0.141667  0.072655
similarité items  1.000000  0.579646


## 2. Graphe et split transductif

Deux communautés structurales et des features bruitées sont générées. Le train reçoit quelques labels seulement. Toutes les arêtes et features sont visibles : aucun label test n’est utilisé dans la loss.

In [2]:
torch.manual_seed(42)
n = 120
labels = np.repeat([0,1],n//2)
prob = np.where(labels[:,None]==labels[None,:],.18,.015)
upper = np.triu(rng.random((n,n)) < prob,k=1)
adj = upper + upper.T
a = torch.tensor(adj,dtype=torch.float32)+torch.eye(n)
degree = a.sum(1)
normalized = degree.rsqrt()[:,None]*a*degree.rsqrt()[None,:]
x = torch.tensor(rng.normal(size=(n,6)),dtype=torch.float32)
x[:,0] += torch.tensor(labels,dtype=torch.float32)*.6
y = torch.tensor(labels,dtype=torch.long)
train_ids = torch.tensor(np.r_[rng.choice(60,12,False),rng.choice(np.arange(60,120),12,False)])
test_mask = torch.ones(n,dtype=torch.bool); test_mask[train_ids] = False
assert torch.isfinite(normalized).all()

## 3. MLP versus GCN

La même taille de réseau et le même budget sont utilisés. Le MLP ignore la structure. La GCN applique deux propagations normalisées. Un graphe non homophile pourrait changer le résultat.

In [3]:
def fit(use_graph):
    torch.manual_seed(7)
    first, second = nn.Linear(6,16), nn.Linear(16,2)
    optimizer = torch.optim.Adam(list(first.parameters())+list(second.parameters()),lr=.02,weight_decay=.01)
    def forward(features):
        hidden = torch.relu(first(normalized@features if use_graph else features))
        return second(normalized@hidden if use_graph else hidden)
    for _ in range(160):
        loss = nn.functional.cross_entropy(forward(x)[train_ids],y[train_ids])
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    with torch.no_grad():
        logits = forward(x)
        return (logits[test_mask].argmax(1)==y[test_mask]).float().mean().item()
print({'MLP test':fit(False),'GCN test transductif':fit(True)})

{'MLP test': 0.625, 'GCN test transductif': 1.0}


## Exercice

Calculez NDCG@3 pour [1,0,1]. Expliquez pourquoi le protocole GCN n’est pas une généralisation à de nouveaux graphes.

In [4]:
# Écrivez votre expérience ici avant de lire la correction.

## Correction et limites

NDCG=(1+1/log2(4))/(1+1/log2(3))≈0,920. Les nœuds test et leurs connexions sont visibles pendant l’apprentissage ; seuls leurs labels sont masqués. Pour l’inductif, réserver des graphes ou nœuds et leurs données selon le scénario de déploiement.